In [26]:
from langchain_ollama.llms import OllamaLLM
llm = OllamaLLM(model="qwen2:1.5b",temperature=0.3)

In [2]:
from langchain.document_loaders import PyPDFLoader

In [6]:
%pwd

'd:\\Quoc Thang\\DATA ANALYST PROJECT\\15 project Generative AI\\Project 1 ( Chatbot)'

In [5]:
%cd ..

d:\Quoc Thang\DATA ANALYST PROJECT\15 project Generative AI\Project 1 ( Chatbot)


d:\Quoc Thang\DATA ANALYST PROJECT\15 project Generative AI\Project 1 ( Chatbot)\env\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [7]:
loader = PyPDFLoader(r'D:\Quoc Thang\DATA ANALYST PROJECT\15 project Generative AI\Project 1 ( Chatbot)\data\SDG.pdf')
data = loader.load()

In [9]:
len(data)

24

In [15]:
question_gen = ""
for page in data:
    question_gen+= page.page_content

In [18]:
from langchain.text_splitter import TokenTextSplitter


In [57]:
splitter_ques_gen = TokenTextSplitter(
    model_name= "gpt-3.5-turbo",
    chunk_size= 10000,
    chunk_overlap = 200
)

In [58]:
chunk_ques_gen = splitter_ques_gen.split_text(question_gen)

In [59]:
chunk_ques_gen

['IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough  \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew  \nthat earthquakes and floods were inevitable, but that the high death  \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence that we can succeed. In the past 15

In [60]:
splitter_ans_gen = TokenTextSplitter(
    model_name= "gpt-3.5-turbo",
    chunk_size= 1000,
    chunk_overlap = 200
)

In [61]:
document_answer_gen = splitter_ans_gen.split_documents(
    document_ques_gen
)

In [62]:
document_answer_gen

[Document(metadata={}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough  \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew  \nthat earthquakes and floods were inevitable, but that the high death  \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence 

In [24]:
from langchain.docstore.document import Document
document_ques_gen = [Document(i) for i in chunk_ques_gen]

In [25]:
document_ques_gen

[Document(metadata={}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough  \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew  \nthat earthquakes and floods were inevitable, but that the high death  \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence 

In [27]:
prompt_template = """
You are an expert at creating questions based on coding materials and documentation.
Your goal is to prepare a coder or programmer for their exam and coding tests.
You do this by asking questions about the text below:

------------
{text}
------------

Create questions that will prepare the coders or programmers for their tests.
Make sure not to lose any important information.

QUESTIONS:
"""

In [28]:
from langchain.prompts import PromptTemplate

In [29]:
PROMTTEMPLATE = PromptTemplate(template=prompt_template ,input_variables=['text'])

In [30]:
refine_template = ("""
You are an expert at creating practice questions based on coding material and documentation.
Your goal is to help a coder or programmer prepare for a coding test.
We have received some practice questions to a certain extent: {existing_answer}.
We have the option to refine the existing questions or add new ones.
(only if necessary) with some more context below.
------------
{text}
------------

Given the new context, refine the original questions in English.
If the context is not helpful, please provide the original questions.
QUESTIONS:
"""
)


In [38]:
REFINE_PROMPT_QUESTIONS = PromptTemplate(
    input_variables=["existing_answer", "text"],
    template=refine_template,
)

In [31]:
from langchain.chains.summarize import load_summarize_chain

In [39]:
ques_gen_chain = load_summarize_chain(llm=llm ,chain_type='refine', verbose = True, 
                                          question_prompt=PROMTTEMPLATE  , 
                                          refine_prompt=REFINE_PROMPT_QUESTIONS)

In [63]:
from openvino.runtime import Core

ie = Core()

In [68]:
devices = ie.available_devices

for device in devices:
    device_name = ie.get_property(device, "FULL_DEVICE_NAME")
    print(f"{device}: {device_name}")

CPU: 11th Gen Intel(R) Core(TM) i5-1135G7 @ 2.40GHz
GPU: Intel(R) Iris(R) Xe Graphics (iGPU)


In [69]:
ques = ques_gen_chain.run(document_ques_gen)

print(ques)



> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

You are an expert at creating questions based on coding materials and documentation.
Your goal is to prepare a coder or programmer for their exam and coding tests.
You do this by asking questions about the text below:

------------
IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD 
CAME TOGETHER TO FACE THE FUTURE.
And what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. 
Not just in some faraway place, but in their own cities and towns and villages.
They knew things didn’t have to be this way. They knew we had enough  
food to feed the world, but that it wasn’t getting shared. They knew there 
were medicines for HIV and other diseases, but they cost a lot. They knew  
that earthquakes and floods were inevitable, but that the high death  
tolls were not. 
They also knew that billions of people worldwide shared their hope for a 
better future.
So leaders f

KeyboardInterrupt: 

In [ ]:
ques

In [41]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="qwen2:1.5b",
)

In [42]:
from langchain.vectorstores import FAISS

In [43]:
vector_store = FAISS.from_documents(documents=document_ques_gen, embedding=embeddings,)

In [44]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2:1.5b",
    temperature=0.1,
)

In [46]:
ques_list = ques.split("\n")

In [51]:
ques_list = [i for i in ques_list if "?" in i]

In [53]:
from langchain.chains import RetrievalQA

In [55]:
answer_generation_chain = RetrievalQA.from_chain_type(llm=llm, 
                                               chain_type="stuff", 
                                               retriever=vector_store.as_retriever())


In [56]:
# Answer each question and save to a file
for question in ques_list:
    print("Question: ", question)
    answer = answer_generation_chain.run(question)
    print("Answer: ", answer)
    print("--------------------------------------------------\\n\\n")
    # Save answer to file
    with open("answers.txt", "a") as f:
        f.write("Question: " + question + "\\n")
        f.write("Answer: " + answer + "\\n")
        f.write("--------------------------------------------------\\n\\n")

Question:  1. What are some of the goals outlined in the text related to sustainable development?
Answer:  The text outlines several key goals for sustainable development, including:

  1. Reducing greenhouse gas emissions and transitioning to renewable energy sources.
  2. Improving access to clean water and sanitation services.
  3. Enhancing biodiversity conservation and protecting natural ecosystems.
  4. Promoting sustainable agriculture practices that reduce environmental impacts and improve food security.
  5. Strengthening disaster risk reduction and management systems.
  6. Encouraging the use of low-carbon technologies in industry and transportation.
  7. Supporting education, training, and research on sustainable development issues.

These goals are aimed at addressing pressing global challenges related to climate change, environmental degradation, and social inequality, while promoting economic growth and sustainable human well-being.
---------------------------------------